# Phase 4: fine-tune on dark-skin images (Fitzpatrick17k) and measure the change honestly

**Decision-support prototype only. Not a diagnosis. Consult a qualified clinician.**

Phase 3 showed that adding more light-skin data made dark skin worse. This notebook tries the direct approach: train on dark-skin images.

What it does, in order:
1. Collects the **Fitzpatrick17k** clinical photos labelled Fitzpatrick IV, V or VI whose diagnosis maps to one of our 7 classes and whose link still works.
2. Splits those images into 5 folds. For each fold, it fine-tunes the original EfficientNet-B3 on HAM10000 plus the other 4 folds, and predicts the held-out fold. Every dark-skin image is therefore predicted by a model that never saw it (cross-validation). This is the honest way to measure on a small dataset.
3. Trains one final model on HAM10000 plus all dark-skin images. This is the candidate for the app.
4. Compares the original and the fine-tuned models on: the dark-skin images (cross-validated), the original HAM10000 test set, and 410 benign dark-skin lesions from the ISIC Archive (test only, never trained on).
5. Applies a promotion rule written before the run. The candidate is exported only if it passes.

**Honest scope.** The dark-skin set is small (a few hundred images at most, web-sourced, some label noise, no patient ids). Results are indicative, with wide confidence intervals. Nothing here shows the model is safe to use on dark skin.

Runs on **Kaggle**: GPU T4, internet on, dataset `kmader/skin-cancer-mnist-ham10000` attached.

## 1. Setup

In [ ]:
import os, sys, io, json, time, random, subprocess, urllib.request, urllib.parse
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image, ImageFile
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score
ImageFile.LOAD_TRUNCATED_IMAGES = True

IN_KAGGLE = os.path.exists("/kaggle/working")
SMOKE = os.environ.get("PHASE4_SMOKE") == "1"      # tiny local test mode

# ---------------- ONE CONFIG BLOCK: every number lives here ----------------
CFG = dict(
    repo_url="https://github.com/Elvis-Kayonga/ai-skin-lesion-detection-system-capstone-project.git",
    seed=42,
    class_codes=["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"],
    high_risk=["akiec", "bcc", "mel"],          # pre-cancer / cancer classes
    input_size=300, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
    dark_fitz_types=[4, 5, 6],   # Fitzpatrick IV-VI
    folds=5,                     # cross-validation folds over the dark-skin images
    epochs=6,                    # FIXED number of epochs (no early stopping on the held-out fold)
    samples_per_epoch=3000, dark_share=0.3,     # each epoch: 70% HAM10000 train, 30% dark-skin train (oversampled)
    batch_size=32, lr_backbone=1e-4, lr_head=3e-4, weight_decay=1e-4,
    mc_passes=10, mc_seed=1234,
    entropy_percentile=80.0, mel_recall_target=0.90,     # same threshold recipe as the training notebook
    bootstrap=1000,
    promote=dict(ham_macro_f1_drop=0.01, ham_mel_sens_drop=0.03, dark_false_alarm_rise=0.02, dark_hr_flag_drop=0.0),
    max_per_source=None, workers=16,
    num_workers=2 if os.path.exists("/kaggle/working") else 0,
)
if SMOKE:
    CFG.update(epochs=1, samples_per_epoch=16, batch_size=4, bootstrap=50, max_per_source=30, workers=8, mc_passes=3, folds=2)

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    (torch.ones(8, device=device) * 2).sum().item()
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU, training will be very slow")
CODES = CFG["class_codes"]; IDX = {c: i for i, c in enumerate(CODES)}
MEL = IDX["mel"]; HR_IDX = [IDX[c] for c in CFG["high_risk"]]

if IN_KAGGLE:
    OUT, WORK = "/kaggle/working/phase4", "/tmp/work"
else:
    OUT = os.environ.get("PHASE4_OUT", "phase4_out"); WORK = os.environ.get("PHASE4_WORK", "phase4_work")
os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)

REPO = os.environ.get("PHASE4_REPO") or f"{WORK}/repo"
if not os.path.exists(f"{REPO}/backend/model/efficientnet_b3_best.pth"):
    subprocess.run(["git", "clone", "--depth", "1", CFG["repo_url"], REPO], check=True)
ORIG_CFG = json.load(open(f"{REPO}/backend/model/model_config.json"))
print("device:", device, "| smoke mode:", SMOKE, "| outputs:", OUT)

In [ ]:
# Small download helpers (retries, browser-like user agent, image check before saving)
UA = {"User-Agent": "Mozilla/5.0 (capstone research; non-commercial)"}

def http_json(url):
    for attempt in range(4):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
                return json.load(r)
        except Exception:
            if attempt == 3: raise
            time.sleep(2 * (attempt + 1))

def isic_search(params):
    """All results of an ISIC Archive image search (public API, no login)."""
    url = "https://api.isic-archive.com/api/v2/images/search/?" + urllib.parse.urlencode({**params, "limit": 100})
    out = []
    while url:
        page = http_json(url); out += page["results"]; url = page.get("next")
    return out

def download_one(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return True
    for attempt in range(3):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=30) as r:
                data = r.read()
            Image.open(io.BytesIO(data)).verify()              # reject HTML error pages / broken files
            with open(path, "wb") as f: f.write(data)
            return True
        except Exception:
            time.sleep(1 + attempt)
    return False

def download_many(df):
    if len(df) == 0:
        return df
    os.makedirs(os.path.dirname(df["path"].iloc[0]), exist_ok=True)
    with ThreadPoolExecutor(CFG["workers"]) as ex:
        ok = list(ex.map(lambda a: download_one(*a), zip(df["url"], df["path"])))
    print(f"   downloaded {sum(ok)} of {len(ok)}")
    return df[np.array(ok, dtype=bool)].reset_index(drop=True)

def limit(df):
    n = CFG["max_per_source"]
    return df if not n or len(df) <= n else df.sample(n, random_state=CFG["seed"]).reset_index(drop=True)

clin = lambda r, k: (r.get("metadata", {}).get("clinical") or {}).get(k)
ext_of = lambda url: os.path.splitext(urllib.parse.urlparse(url).path)[1] or ".jpg"

## 2. Data

| Source | Use | Licence |
|---|---|---|
| Fitzpatrick17k, Fitzpatrick IV-VI, labels mapped to our 7 classes | **train and test by 5-fold cross-validation** | CC BY-NC-SA 3.0 (Groh et al. 2021) |
| HAM10000 (Kaggle) | train (original frozen split), val for thresholds, test | CC BY-NC 4.0 |
| ISIC Archive, Fitzpatrick V-VI, Memorial Sloan Kettering, benign | **test only** (false alarms) | CC BY |

Fitzpatrick17k has no patient ids, so each image is its own group (duplicates removed by file hash). Images are never committed to the repo.

In [ ]:
# ---- Dark skin, labelled: Fitzpatrick17k IV-VI, labels that map to our classes (same mapping as Phase 3) ----
FITZ_MAP = {"melanoma": "mel", "superficial spreading melanoma ssm": "mel", "malignant melanoma": "mel",
            "lentigo maligna": "mel", "basal cell carcinoma": "bcc", "solid cystic basal cell carcinoma": "bcc",
            "basal cell carcinoma morpheiform": "bcc", "actinic keratosis": "akiec", "seborrheic keratosis": "bkl",
            "nevocytic nevus": "nv", "halo nevus": "nv", "congenital nevus": "nv", "dermatofibroma": "df",
            "pyogenic granuloma": "vasc", "granuloma pyogenic": "vasc"}
fz_all = pd.read_csv("https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv")
fz_all = fz_all[fz_all["label"].isin(FITZ_MAP)].drop_duplicates("md5hash")
def fitz_frame(d, name):
    return pd.DataFrame(dict(image_id=d["md5hash"], url=d["url"], path=[f"{WORK}/fitz/{h}.jpg" for h in d["md5hash"]],
                             dx=d["label"].map(FITZ_MAP), group=d["md5hash"], fitz=d["fitzpatrick_scale"], source=name))
dark = fitz_frame(fz_all[fz_all["fitzpatrick_scale"].isin(CFG["dark_fitz_types"])], "Fitzpatrick17k IV-VI")
print("Fitzpatrick17k IV-VI candidates:", len(dark), dark["dx"].value_counts().to_dict())
dark = download_many(limit(dark))
print("Fitzpatrick17k IV-VI usable:", len(dark), dark["dx"].value_counts().to_dict(), "| by type:", dark["fitz"].value_counts().to_dict())

# ---- Dark skin, benign (TEST ONLY): ISIC Archive images labelled Fitzpatrick V-VI (same set as Phase 3) ----
dark_recs = isic_search({"query": "fitzpatrick_skin_type:V OR fitzpatrick_skin_type:VI"})
rows = []
for r in sorted(dark_recs, key=lambda r: r["isic_id"]):
    if (r.get("attribution") or "").startswith("Memorial Sloan Kettering") and clin(r, "diagnosis_1") == "Benign":
        url = r["files"]["full"]["url"]
        rows.append(dict(image_id=r["isic_id"], url=url, path=f"{WORK}/dark_benign/{r['isic_id']}{ext_of(url)}",
                         dx=None, lesion=clin(r, "lesion_id"), group=clin(r, "patient_id"),
                         fitz=clin(r, "fitzpatrick_skin_type"), source="ISIC Fitzpatrick V-VI benign (MSK)"))
dark_benign = download_many(limit(pd.DataFrame(rows).drop_duplicates("lesion")))
print("dark-skin benign test set:", len(dark_benign), "lesions |", dark_benign["group"].nunique(), "patients")

In [ ]:
# ---- HAM10000 with the ORIGINAL frozen split (from the repo) ----
splits = {n: pd.read_csv(f"{REPO}/docs/results/split_{n}.csv") for n in ["train", "val", "test"]}
if IN_KAGGLE:
    import glob
    index = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob("/kaggle/input/**/*.jpg", recursive=True)}
    print("HAM10000 images found:", len(index))
    def ham(name):
        d = splits[name].copy(); d["path"] = d["image_id"].map(index)
        assert d["path"].notna().all(), "some HAM10000 images are missing"
        d["group"] = d["lesion_id"]; d["source"] = "HAM10000"; d["fitz"] = None
        return limit(d)
    ham_train, ham_val, ham_test = ham("train"), ham("val"), ham("test")
    orig_counts = pd.read_csv(f"{REPO}/docs/results/split_class_counts.csv").set_index("dx")
    same = all(int(splits[n]["dx"].value_counts().get(c, 0)) == int(orig_counts.loc[c, n]) for n in splits for c in orig_counts.index)
    assert same, "HAM10000 split does not match the original run: stop and check docs/results/split_*.csv"
else:
    # Local smoke test only: HAM10000 is not on this machine, so Fitzpatrick17k I-III images stand in for it.
    print("LOCAL SMOKE MODE: Fitzpatrick17k I-III images stand in for HAM10000 (results meaningless).")
    light = download_many(limit(fitz_frame(fz_all[fz_all["fitzpatrick_scale"].isin([1, 2, 3])], "HAM10000 (smoke stand-in)")))
    third = len(light) // 3
    ham_train, ham_val, ham_test = light.iloc[:third], light.iloc[third:2 * third], light.iloc[2 * third:]
for d in (ham_train, ham_val, ham_test, dark):
    d["label"] = d["dx"].map(IDX).astype(int)
dark_benign["label"] = -1
print("HAM10000:", {k: len(v) for k, v in [("train", ham_train), ("val", ham_val), ("test", ham_test)]})

# ---- 5-fold split of the dark-skin images (stratified by class) ----
sgkf = StratifiedGroupKFold(n_splits=CFG["folds"], shuffle=True, random_state=CFG["seed"])
dark["fold"] = -1
for k, (_, te) in enumerate(sgkf.split(dark, dark["dx"], groups=dark["group"])):
    dark.loc[dark.index[te], "fold"] = k
assert (dark["fold"] >= 0).all()
manifest = dict(dark_fitz17k=dict(n=len(dark), classes=dark["dx"].value_counts().to_dict(),
                                  fitzpatrick=dark["fitz"].value_counts().to_dict(), folds=dark["fold"].value_counts().to_dict()),
                dark_benign_test=dict(n=len(dark_benign), patients=int(dark_benign["group"].nunique())),
                ham=dict(train=len(ham_train), val=len(ham_val), test=len(ham_test)),
                licences={"HAM10000": "CC BY-NC 4.0", "Fitzpatrick17k": "CC BY-NC-SA 3.0 (Groh et al. 2021)",
                          "ISIC Fitzpatrick V-VI (MSK)": "CC BY (Memorial Sloan Kettering Cancer Center)"})
json.dump(manifest, open(f"{OUT}/phase4_data_manifest.json", "w"), indent=2, default=str)
dark[["image_id", "dx", "fitz", "fold"]].to_csv(f"{OUT}/phase4_dark_folds.csv", index=False)
print(json.dumps(manifest, indent=1, default=str))

## 3. Model, inference and metrics
Same inference contract as the app: resize straight to 300x300, ImageNet normalisation, backbone once, classifier run 10 times with dropout on (MC Dropout), average the softmax.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((CFG["input_size"], CFG["input_size"])),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(30),
    transforms.RandomResizedCrop(CFG["input_size"], scale=(0.7, 1.0), ratio=(0.9, 1.1)),    # clinical photos are framed loosely
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(), transforms.Normalize(CFG["mean"], CFG["std"])])
eval_tf = transforms.Compose([transforms.Resize((CFG["input_size"], CFG["input_size"])),
                              transforms.ToTensor(), transforms.Normalize(CFG["mean"], CFG["std"])])

class ImgSet(Dataset):
    def __init__(self, df, tf): self.paths, self.labels, self.tf = df["path"].values, df["label"].values, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB")), int(self.labels[i])

def build_model():
    m = models.efficientnet_b3(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CODES))
    return m

def load_original():
    m = build_model()
    m.load_state_dict(torch.load(f"{REPO}/backend/model/efficientnet_b3_best.pth", map_location="cpu"))
    return m.to(device).eval()

@torch.no_grad()
def predict(model, df):
    """Returns (MC Dropout averaged probabilities, plain eval-mode probabilities)."""
    torch.manual_seed(CFG["mc_seed"])
    P_mc, P_plain = [], []
    model.eval()
    for x, _ in DataLoader(ImgSet(df, eval_tf), batch_size=32, shuffle=False, num_workers=CFG["num_workers"]):
        feats = F.adaptive_avg_pool2d(model.features(x.to(device)), 1).flatten(1)
        P_plain.append(F.softmax(model.classifier(feats), 1).cpu())
        model.classifier.train()                                   # dropout ON (classifier only)
        P_mc.append(torch.stack([F.softmax(model.classifier(feats), 1) for _ in range(CFG["mc_passes"])]).mean(0).cpu())
        model.classifier.eval()
    return torch.cat(P_mc).numpy(), torch.cat(P_plain).numpy()

entropy = lambda P: -(P * np.log(P + 1e-12)).sum(1)

def thresholds_from_val(P_val, y_val):
    """Same recipe as the training notebook, chosen on HAM10000 VALIDATION only."""
    ent = float(np.percentile(entropy(P_val), CFG["entropy_percentile"]))
    is_mel = y_val == MEL
    ok = [t for t in np.linspace(0.05, 0.95, 91) if is_mel.any() and (P_val[is_mel, MEL] >= t).mean() >= CFG["mel_recall_target"]]
    return ent, float(max(ok)) if ok else 0.05

def cluster_ci(fn, groups, n_boot):
    """95% CI by resampling whole groups (patients, lesions or images)."""
    groups = np.asarray(groups); uniq = np.unique(groups); pos = {g: np.where(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(CFG["seed"]); vals = []
    for _ in range(n_boot):
        idx = np.concatenate([pos[g] for g in rng.choice(uniq, len(uniq), replace=True)])
        v = fn(idx)
        if v is not None and np.isfinite(v): vals.append(v)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def evaluate(P_mc, P_plain, df, ent_thr, mel_thr, flag=None):
    """Labelled sets: accuracy/F1/sensitivities. Benign-only set: false-alarm rates. All with 95% CIs.
    `flag` can be passed in when each image has its own thresholds (cross-validation)."""
    pred = P_plain.argmax(1)
    if flag is None: flag = (entropy(P_mc) >= ent_thr) | (P_mc[:, MEL] >= mel_thr)
    y = df["label"].values; g = df["group"].values; out = {"n": int(len(df)), "patients_or_groups": int(len(np.unique(g)))}
    def add(name, fn):
        v = fn(np.arange(len(y))); lo, hi = cluster_ci(fn, g, CFG["bootstrap"])
        out[name] = None if v is None else round(float(v), 4); out[name + "_ci95"] = [round(lo, 4), round(hi, 4)]
    if (y >= 0).all():
        labels = np.unique(y); hr_true = np.isin(y, HR_IDX)
        add("accuracy", lambda i: float((pred[i] == y[i]).mean()))
        add("macro_f1", lambda i: float(f1_score(y[i], pred[i], labels=labels, average="macro", zero_division=0)))
        add("melanoma_sensitivity", lambda i: float((pred[i][y[i] == MEL] == MEL).mean()) if (y[i] == MEL).any() else None)
        add("high_risk_sensitivity", lambda i: float(np.isin(pred[i][hr_true[i]], HR_IDX).mean()) if hr_true[i].any() else None)
        add("high_risk_sensitivity_with_flag",
            lambda i: float((np.isin(pred[i], HR_IDX) | flag[i])[hr_true[i]].mean()) if hr_true[i].any() else None)
        add("benign_false_alarm_rate",
            lambda i: float(np.isin(pred[i][~hr_true[i]], HR_IDX).mean()) if (~hr_true[i]).any() else None)
        add("review_flag_rate", lambda i: float(flag[i].mean()))
    else:
        add("false_alarm_high_risk_rate", lambda i: float(np.isin(pred[i], HR_IDX).mean()))
        add("false_alarm_melanoma_rate", lambda i: float((pred[i] == MEL).mean()))
        add("review_flag_rate", lambda i: float(flag[i].mean()))
    return out

## 4. Fine-tuning recipe
Starts from the original weights with small learning rates. Each epoch draws 70% of its samples from HAM10000 train and 30% from the dark-skin training folds (the dark images are repeated, with augmentation, because there are few of them). The number of epochs is fixed in advance, so the held-out fold is never used to pick a model.

In [ ]:
ham_counts = np.bincount(ham_train["label"].values, minlength=len(CODES)).astype(float); ham_counts[ham_counts == 0] = 1
class_w = torch.tensor(ham_counts.sum() / (len(CODES) * ham_counts), dtype=torch.float32)

def finetune(dark_train, tag):
    train_df = pd.concat([ham_train, dark_train], ignore_index=True)
    is_dark = np.r_[np.zeros(len(ham_train), bool), np.ones(len(dark_train), bool)]
    w = np.where(is_dark, CFG["dark_share"] / max(is_dark.sum(), 1), (1 - CFG["dark_share"]) / max((~is_dark).sum(), 1))
    model = load_original().train()
    crit = nn.CrossEntropyLoss(weight=class_w.to(device))
    opt = torch.optim.AdamW([{"params": model.features.parameters(), "lr": CFG["lr_backbone"]},
                             {"params": model.classifier.parameters(), "lr": CFG["lr_head"]}], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
    loader = DataLoader(ImgSet(train_df, train_tf), batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], drop_last=True,
                        sampler=WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=CFG["samples_per_epoch"], replacement=True))
    hist = []
    for ep in range(CFG["epochs"]):
        model.train(); t0, tot, n = time.time(), 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); tot += float(loss) * len(y); n += len(y)
        sched.step()
        hist.append(dict(model=tag, epoch=ep + 1, train_loss=tot / max(n, 1)))
        print(f"[{tag}] ep {ep+1}/{CFG['epochs']} loss {tot/max(n,1):.4f} | {time.time()-t0:.0f}s")
    model.eval()
    P_val, _ = predict(model, ham_val)
    return model, thresholds_from_val(P_val, ham_val["label"].values), hist

## 5. Original model on every evaluation set (baseline)

In [ ]:
orig = load_original()
ORIG_THR = (ORIG_CFG["entropy_threshold"], ORIG_CFG["melanoma_flag_threshold"])   # what the app uses today
P_orig_dark = predict(orig, dark)
results, preds = [], {}
def record(model_name, set_name, P, d, thr):
    r = evaluate(P[0], P[1], d, *thr); r.update(model=model_name, eval_set=set_name); results.append(r)
    print(model_name, "|", set_name, {k: v for k, v in r.items() if not k.endswith("_ci95")})
vi = dark["fitz"].isin([5, 6]).values
record("original", "Dark skin IV-VI (Fitzpatrick17k)", P_orig_dark, dark, ORIG_THR)
record("original", "Dark skin V-VI only (Fitzpatrick17k)", (P_orig_dark[0][vi], P_orig_dark[1][vi]), dark[vi], ORIG_THR)
record("original", "HAM10000 test", predict(orig, ham_test), ham_test, ORIG_THR)
record("original", "Dark skin V-VI benign (ISIC/MSK)", predict(orig, dark_benign), dark_benign, ORIG_THR)
del orig; torch.cuda.empty_cache()

## 6. Cross-validation on the dark-skin images
Each fold model is trained without its fold, then predicts only that fold. Joining the 5 held-out folds gives one honest prediction for every dark-skin image.

In [ ]:
oof_mc = np.zeros((len(dark), len(CODES))); oof_plain = np.zeros_like(oof_mc); flags_thr = np.zeros((len(dark), 2))
hist_all = []
for k in range(CFG["folds"]):
    tr, te = dark[dark["fold"] != k], dark[dark["fold"] == k]
    m, thr, h = finetune(tr, f"fold{k}"); hist_all += h
    P_mc, P_plain = predict(m, te)
    pos = np.where(dark["fold"].values == k)[0]
    oof_mc[pos], oof_plain[pos], flags_thr[pos] = P_mc, P_plain, thr
    print(f"fold {k}: {len(te)} held-out images, accuracy {(P_plain.argmax(1) == te['label'].values).mean():.3f}")
    del m; torch.cuda.empty_cache()

# Each image is flagged with the thresholds of the fold model that predicted it
cv_flag = (entropy(oof_mc) >= flags_thr[:, 0]) | (oof_mc[:, MEL] >= flags_thr[:, 1])
def record_cv(set_name, mask):
    r = evaluate(oof_mc[mask], oof_plain[mask], dark[mask], None, None, flag=cv_flag[mask])
    r.update(model="fine-tuned (cross-validated)", eval_set=set_name); results.append(r)
    print("fine-tuned (CV) |", set_name, {k: v for k, v in r.items() if not k.endswith("_ci95")})
pd.DataFrame(dict(image_id=dark["image_id"], dx=dark["dx"], fitz=dark["fitz"], fold=dark["fold"],
                  original_pred=[CODES[i] for i in P_orig_dark[1].argmax(1)], finetuned_cv_pred=[CODES[i] for i in oof_plain.argmax(1)],
                  finetuned_cv_flag=cv_flag)).to_csv(f"{OUT}/phase4_dark_predictions.csv", index=False)
record_cv("Dark skin IV-VI (Fitzpatrick17k)", np.ones(len(dark), bool))
record_cv("Dark skin V-VI only (Fitzpatrick17k)", vi)

## 7. Final model: HAM10000 + all dark-skin images
This is the candidate for the app. Its dark-skin performance is the cross-validated estimate above (it has seen every dark-skin image, so it cannot be tested on them). It is tested on HAM10000 test and the ISIC benign dark-skin set, which it never saw.

In [ ]:
final, FT_THR, h = finetune(dark, "final"); hist_all += h
print(f"final thresholds (HAM10000 validation): entropy {FT_THR[0]:.4f}, melanoma {FT_THR[1]:.2f}")
record("fine-tuned (final)", "HAM10000 test", predict(final, ham_test), ham_test, FT_THR)
record("fine-tuned (final)", "Dark skin V-VI benign (ISIC/MSK)", predict(final, dark_benign), dark_benign, FT_THR)
pd.DataFrame(hist_all).to_csv(f"{OUT}/phase4_finetune_history.csv", index=False)

res = pd.DataFrame(results)
res.to_csv(f"{OUT}/phase4_eval.csv", index=False)
show = [c for c in ["accuracy", "macro_f1", "melanoma_sensitivity", "high_risk_sensitivity", "high_risk_sensitivity_with_flag",
                    "benign_false_alarm_rate", "false_alarm_high_risk_rate", "review_flag_rate"] if c in res]
pd.set_option("display.width", 220)
print(res.pivot_table(index="eval_set", columns="model", values=show, aggfunc="first").round(3))

## 8. Promotion rule (written before the run)
The final model replaces the app's model ONLY if all of these hold:
1. HAM10000 test macro-F1 drops by no more than 0.01.
2. HAM10000 test melanoma sensitivity drops by no more than 0.03.
3. Dark-skin IV-VI accuracy (cross-validated) is higher than the original model's on the same images.
4. Dark-skin IV-VI high-risk sensitivity counting the review flag does not drop.
5. False alarms on the 410 benign ISIC dark-skin lesions rise by no more than 0.02.

In [ ]:
def get(model, s, m):
    row = res[(res.model == model) & (res.eval_set == s)]
    return float(row[m].iloc[0]) if len(row) and m in row and pd.notna(row[m].iloc[0]) else np.nan
P = CFG["promote"]; H, D, B = "HAM10000 test", "Dark skin IV-VI (Fitzpatrick17k)", "Dark skin V-VI benign (ISIC/MSK)"
FT, CV = "fine-tuned (final)", "fine-tuned (cross-validated)"
checks = {
    "HAM macro-F1 not worse":          get(FT, H, "macro_f1") >= get("original", H, "macro_f1") - P["ham_macro_f1_drop"],
    "HAM melanoma sens. not worse":    get(FT, H, "melanoma_sensitivity") >= get("original", H, "melanoma_sensitivity") - P["ham_mel_sens_drop"],
    "dark-skin accuracy improved":     get(CV, D, "accuracy") > get("original", D, "accuracy"),
    "dark-skin high-risk catch not down": get(CV, D, "high_risk_sensitivity_with_flag") >= get("original", D, "high_risk_sensitivity_with_flag") - P["dark_hr_flag_drop"],
    "ISIC dark-skin false alarms not up": get(FT, B, "false_alarm_high_risk_rate") <= get("original", B, "false_alarm_high_risk_rate") + P["dark_false_alarm_rise"],
}
PROMOTE = all(bool(v) for v in checks.values()) and not SMOKE
for k, v in checks.items(): print(("PASS " if v else "FAIL ") + k)
print("\nPROMOTE TO APP:", PROMOTE, "(smoke mode never promotes)" if SMOKE else "")

In [ ]:
# Summary + candidate model files with the SAME contract as backend/model/ (only if promoted)
def contract_metrics(model, d, thr):
    P_mc, P = predict(model, d); y = d["label"].values; pred = P.argmax(1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    try: auc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(len(CODES)))))
    except ValueError: auc = float("nan")
    flag = (entropy(P_mc) >= thr[0]) | (P_mc[:, MEL] >= thr[1]); mc_pred = P_mc.argmax(1); ok = mc_pred == y; mel = y == MEL
    test = dict(accuracy=float((pred == y).mean()), macro_precision=float(pr), macro_recall=float(rc), macro_f1=float(f1),
                macro_auc_ovr=auc, melanoma_sensitivity=float((pred[mel] == MEL).mean()) if mel.any() else None)
    mission = dict(mc_accuracy=float(ok.mean()), review_rate=float(flag.mean()),
                   accuracy_unflagged=float(ok[~flag].mean()) if (~flag).any() else None,
                   accuracy_flagged=float(ok[flag].mean()) if flag.any() else None,
                   errors_caught_by_flag=float(flag[~ok].mean()) if (~ok).any() else None,
                   melanoma_sensitivity_argmax=float(ok[mel].mean()) if mel.any() else None,
                   melanoma_sensitivity_with_flag=float((ok | flag)[mel].mean()) if mel.any() else None)
    return test, mission

test, mission = contract_metrics(final, ham_test, FT_THR)
summary = dict(promote=PROMOTE, checks={k: bool(v) for k, v in checks.items()}, finetuned_thresholds=FT_THR,
               original_thresholds=ORIG_THR, config={k: v for k, v in CFG.items() if k != "repo_url"},
               finetuned_ham_test_metrics=test, finetuned_ham_mission_metrics=mission,
               honest_scope="Dark-skin training and test data is a small web-sourced Fitzpatrick17k sample (no patient ids, some "
                            "label noise). Cross-validated results are indicative only. Not a validation for clinical use on dark skin.")
json.dump(summary, open(f"{OUT}/phase4_summary.json", "w"), indent=2, default=str)
if PROMOTE:
    os.makedirs(f"{OUT}/model_candidate", exist_ok=True)
    cand = {k: v for k, v in ORIG_CFG.items() if k not in ("temperature", "calibrated_entropy_threshold",
                                                           "calibrated_melanoma_threshold", "calibration")}
    cand.update(model_name="efficientnet_b3_finetuned_ham_fitz17k_dark", entropy_threshold=FT_THR[0],
                melanoma_flag_threshold=FT_THR[1], test_metrics=test, mission_metrics=mission,
                training_data="HAM10000 + Fitzpatrick17k Fitzpatrick IV-VI (fine-tuned from the original EfficientNet-B3)",
                limitations="Fine-tuned with a small web-sourced dark-skin sample (Fitzpatrick17k IV-VI). "
                            "Dark-skin results are cross-validated estimates on few images. Not validated for clinical use on dark skin.")
    torch.save(final.state_dict(), f"{OUT}/model_candidate/efficientnet_b3_best.pth")
    json.dump(cand, open(f"{OUT}/model_candidate/model_config.json", "w"), indent=2)
    print("candidate model written to", f"{OUT}/model_candidate (copy into backend/model/ only after review)")
else:
    print("No candidate exported: the app keeps the original model.")

In [ ]:
# One chart for the README / report
import matplotlib.pyplot as plt
plot_rows = [("Dark skin IV-VI (Fitzpatrick17k)", "accuracy", "fine-tuned (cross-validated)"),
             ("Dark skin IV-VI (Fitzpatrick17k)", "high_risk_sensitivity_with_flag", "fine-tuned (cross-validated)"),
             ("Dark skin V-VI only (Fitzpatrick17k)", "accuracy", "fine-tuned (cross-validated)"),
             ("HAM10000 test", "macro_f1", "fine-tuned (final)"),
             ("Dark skin V-VI benign (ISIC/MSK)", "false_alarm_high_risk_rate", "fine-tuned (final)")]
fig, ax = plt.subplots(figsize=(11, 4)); xs = np.arange(len(plot_rows))
for k, off in enumerate([-0.18, 0.18]):
    vals, los, his = [], [], []
    for s, m, ftname in plot_rows:
        row = res[(res.model == ("original" if k == 0 else ftname)) & (res.eval_set == s)]
        v = float(row[m].iloc[0]) if len(row) and pd.notna(row[m].iloc[0]) else np.nan
        lo, hi = row[m + "_ci95"].iloc[0] if len(row) else (np.nan, np.nan)
        vals.append(v); los.append(max(v - lo, 0)); his.append(max(hi - v, 0))
    ax.bar(xs + off, vals, 0.36, yerr=[los, his], capsize=4, label="original" if k == 0 else "fine-tuned on dark skin")
ax.set_xticks(xs); ax.set_xticklabels([f"{s}\n{m}" for s, m, _ in plot_rows], fontsize=7)
ax.set_ylim(0, 1); ax.legend(); ax.set_title("Original vs fine-tuned on dark skin (95% CI). Lower is better for false alarms.")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_phase4_comparison.png", dpi=150)
print("outputs:", sorted(os.listdir(OUT)))

### After the run
1. Copy `phase4_*.csv/json` into `docs/results/` and `fig_phase4_comparison.png` into `docs/figures/`.
2. If `model_candidate/` exists (the rule passed), review the numbers first, then decide whether to copy its two files into `backend/model/`.
3. Report the results honestly, including any FAIL.